In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_catalogo = spark.table(f"{catalogo}.{esquema_source}.catalogo_cliente_producto")


In [0]:
df_catalogo = df_catalogo.withColumn(
    "llave",
    concat_ws(".", df_catalogo["id_cliente"].cast("string"), df_catalogo["id_producto"].cast("string"))
)

In [0]:
df_catalogo.display()

In [0]:
df_catalogo = df_catalogo.dropna(how="all")\
                        .filter((col("llave").isNotNull()) | (col("cliente").isNotNull()) | (col("producto").isNotNull()))
df_catalogo = df_catalogo.dropDuplicates()
df_catalogo.display()



In [0]:
df_catalogo.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.catalogo_transformed")

In [0]:
%sql
SELECT * FROM catalog_au.silver.catalogo_transformed